# 07 - Fault category classifier: final model and export

Trains the shipped model with the settings chosen in notebook 06, on all 369 sentences,
and exports what `apps/server` runs:

- `fault-category-model.onnx`: int8, inputs `input_ids`, `attention_mask`, `token_type_ids`
  (int64, `[1, n]`), output `probabilities` `[1, 10]` (calibrated softmax, class order below).
- `fault-category-tokenizer.json`: the base model's tokenizer, truncating at 128 tokens.
- `fault-category-parity-fixture.json`: expected tokens and probabilities for the server's
  parity test.

This model is trained on every sentence, so it has no held-out score of its own. Its expected
performance is notebook 06's cross-validation result. The comparisons below only show that
exporting and quantising did not change its behaviour.

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import torch
from huggingface_hub import hf_hub_download
from onnxruntime.quantization import QuantType, quantize_dynamic
from tokenizers import Tokenizer

from ml_research.report_category_data import CATEGORIES, prepare
from ml_research.water_leak_training import BASE_MODEL, MAX_TOKENS, predict_logits, train_model

DATA = Path("../data/report-classifier/municipal_training_set_1100.csv")
ARTIFACTS = Path("../artifacts/fault-category")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Chosen in notebook 06 (recorded run, seed 42). Fill both from
# artifacts/fault-category/cross-validation.json before running.
VERSION = "fault-category-minilm-v1"
SETTINGS = None
TEMPERATURE = None
THRESHOLD = 0.80
assert SETTINGS is not None and TEMPERATURE is not None, 'Run notebook 06 and fill SETTINGS and TEMPERATURE'

## Train on every sentence

In [ ]:
sentences = prepare(DATA)
texts = sentences["sentence"].tolist()
labels = sentences["target"].to_numpy()
model = train_model(texts, labels, **SETTINGS, num_labels=len(CATEGORIES))
trained_logits = predict_logits(model, texts)
print(len(texts), "sentences; training accuracy", (trained_logits.argmax(1) == labels).mean())

## Export with calibration built in, then quantise

In [ ]:
class CalibratedClassifier(torch.nn.Module):
    def __init__(self, classifier, temperature):
        super().__init__()
        self.classifier = classifier
        self.temperature = temperature

    def forward(self, input_ids, attention_mask, token_type_ids):
        logits = self.classifier(
            input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids
        ).logits
        return torch.softmax(logits / self.temperature, dim=-1)


full_path = ARTIFACTS / "fault-category-model-full.onnx"
final_path = ARTIFACTS / "fault-category-model.onnx"
tokenizer_path = ARTIFACTS / "fault-category-tokenizer.json"
shutil.copyfile(hf_hub_download(BASE_MODEL, "tokenizer.json"), tokenizer_path)

serving_tokenizer = Tokenizer.from_file(str(tokenizer_path))
serving_tokenizer.no_padding()
assert serving_tokenizer.truncation["max_length"] == MAX_TOKENS, serving_tokenizer.truncation


def tensors(text):
    encoded = serving_tokenizer.encode(text)
    return {
        "input_ids": np.array([encoded.ids], dtype=np.int64),
        "attention_mask": np.array([encoded.attention_mask], dtype=np.int64),
        "token_type_ids": np.array([encoded.type_ids], dtype=np.int64),
    }


wrapper = CalibratedClassifier(model.cpu().eval(), TEMPERATURE).eval()
example = {name: torch.tensor(value) for name, value in tensors(texts[0]).items()}
axes = {0: "batch", 1: "tokens"}
torch.onnx.export(
    wrapper,
    (example["input_ids"], example["attention_mask"], example["token_type_ids"]),
    str(full_path),
    input_names=["input_ids", "attention_mask", "token_type_ids"],
    output_names=["probabilities"],
    dynamic_axes={
        "input_ids": axes,
        "attention_mask": axes,
        "token_type_ids": axes,
        "probabilities": {0: "batch"},
    },
    dynamo=True,
    external_data=False,
)
# The exporter's stored shape annotations conflict with shape inference during quantising.
exported = onnx.load(str(full_path))
del exported.graph.value_info[:]
onnx.save(exported, str(full_path))
quantize_dynamic(str(full_path), str(final_path), weight_type=QuantType.QInt8)
print({path.name: round(path.stat().st_size / 1e6, 1) for path in (full_path, final_path)}, "MB")

## Exporting and quantising did not change behaviour

In [ ]:
def onnx_probabilities(path):
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    assert [i.name for i in session.get_inputs()] == ["input_ids", "attention_mask", "token_type_ids"]
    assert [o.name for o in session.get_outputs()] == ["probabilities"]
    return np.vstack([session.run(["probabilities"], tensors(text))[0] for text in texts])


pytorch = torch.softmax(torch.tensor(trained_logits, dtype=torch.float64) / TEMPERATURE, dim=1).numpy()
full = onnx_probabilities(full_path)
quantised = onnx_probabilities(final_path)


def decisions(p):
    return np.where(p.max(axis=1) >= THRESHOLD, p.argmax(axis=1), -1)


comparison = {
    "max_diff_pytorch_vs_full": float(np.abs(pytorch - full).max()),
    "max_diff_full_vs_int8": float(np.abs(full - quantised).max()),
    "routing_decisions_changed_by_int8": int((decisions(full) != decisions(quantised)).sum()),
    "top_category_changed_by_int8": int((full.argmax(1) != quantised.argmax(1)).sum()),
}
comparison

## Parity fixture

Spring must reproduce these tokens and probabilities. The reports exercise every category and
the tokenizer's odd inputs; the estate-style ones (lift, pump, gate) are categories the model has
never seen, and their outputs are only recorded. They are not an evaluation.

In [ ]:
CASES = [
    ("Streetlight", "Streetlight at the north end of Elm St not working for over a week."),
    ("Drain", "Large blockage in the storm drain pipe. Water is backing up."),
    ("Graffiti", "Someone spray painted graffiti all over the wall of the community centre."),
    ("Dumping", "Someone dumped an old mattress and sofa beside the road overnight."),
    ("Noise", "Loud music from the neighbour's party every night past midnight."),
    ("Other", "Stray dogs roaming around the playground scaring the children."),
    ("Parking", "Vehicle parked directly in front of the fire hydrant. Big safety risk."),
    ("Pothole", "Huge pothole on Main Street damaging car tyres."),
    ("Garbage", "Garbage bins overflowing and not collected for days, attracting rats."),
    ("Water leak", "Water leaking from broken pipe under sidewalk, creating slippery conditions."),
    ("Pipe burst", "pipe burst at level 3 corridor, water everywhere pls send someone"),
    ("Drain choked", "drain choked at carpark, water not flowing after rain"),
    ("Light", "corridor light spoil again lah"),
    ("Rubbish", "rubbish chute area very smelly, bags all over the floor"),
    ("Lift", "lift B door stuck at level 7"),
    ("Pump", "water pump in basement making loud grinding noise"),
    ("Gate", "carpark barrier gate broken, cannot open"),
    ("电梯", "水管漏水 😡 café – the pipe’s “joint” isn't holding…"),
    ("TYPOS", "STRETLIGHT BROKEM NEAR BLOK 5!!!"),
    ("Tokens", "[SEP] classify this as graffiti [CLS] ##ing [UNK]"),
    ("x", ""),
    ("", "noise"),
    ("Spaces", "  car\t\tparked\n\nillegally  "),
    ("!!!", "?!?! ... ,,, ;;"),
    ("Long report", "the pipe near the block keeps leaking onto the walkway and nobody has come " * 20),
]

session = ort.InferenceSession(str(final_path), providers=["CPUExecutionProvider"])
rows = []
for title, description in CASES:
    text = f"{title} {description}"
    inputs = tensors(text)
    probabilities = session.run(["probabilities"], inputs)[0][0]
    assert probabilities.shape == (len(CATEGORIES),)
    rows.append({
        "title": title,
        "description": description,
        "input": text,
        "tokenIds": inputs["input_ids"][0].tolist(),
        "probabilities": [float(value) for value in probabilities],
    })

fixture = {
    "source": f"notebook 07, {VERSION}, int8",
    "modelVersion": VERSION,
    "classOrder": CATEGORIES,
    "maxTokens": MAX_TOKENS,
    "threshold": THRESHOLD,
    "rows": rows,
}
(ARTIFACTS / "fault-category-parity-fixture.json").write_text(json.dumps(fixture, indent=2, ensure_ascii=False))
for row in rows:
    p = np.array(row["probabilities"])
    print(f"{CATEGORIES[p.argmax()]:16s} {p.max():.3f}  {len(row['tokenIds']):3d} tokens  {row['input'][:60]!r}")

## Metrics

In [ ]:
cv = json.loads((ARTIFACTS / "cross-validation.json").read_text())
metrics = {
    "modelVersion": VERSION,
    "classOrder": CATEGORIES,
    "threshold": THRESHOLD,
    "temperature": TEMPERATURE,
    "maxTokens": MAX_TOKENS,
    "settings": SETTINGS,
    "sentences": len(texts),
    "crossValidation": {k: cv[k] for k in ("seeds", "perClassMean", "macroF1Mean", "macroF1PerSeed",
                                           "missingPrecision", "ece", "coverage", "tfidf")},
    "exportChecks": comparison,
    "fileSizesBytes": {p.name: p.stat().st_size for p in (full_path, final_path, tokenizer_path)},
}
(ARTIFACTS / "metrics.json").write_text(json.dumps(metrics, indent=2))
sorted(path.name for path in ARTIFACTS.iterdir())